# Fraud Detection - Colab Training and Evaluation

Use this notebook when the local PC is too limited for training. It runs the same project code as local training, then exports the trained model artifacts and documentation needed for the final report.

Expected outputs:
- `models/best_model.joblib`
- `models/feature_columns.json`
- `models/metrics.json`
- `models/decision_threshold.json`
- `models/reference_stats.json`
- zipped model artifacts and documentation in Google Drive

## 1. Mount Drive and choose project source

Recommended workflow:
1. Push the latest repo to GitHub.
2. Set `GITHUB_REPO_URL` below.
3. Run all cells.

Fallback workflow:
- Leave `GITHUB_REPO_URL` empty.
- Upload a zip containing the `fraud-detection/` folder when Colab asks.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

from google.colab import drive, files

drive.mount('/content/drive')

# Option A: set your GitHub repo URL, for example:
# GITHUB_REPO_URL = 'https://github.com/<user>/<repo>.git'
GITHUB_REPO_URL = ''

PROJECT_DIR = Path('/content/fraud-detection')
EXPORT_DIR = Path('/content/drive/MyDrive/fraud-detection-mlops-exports')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Project dir:', PROJECT_DIR)
print('Export dir:', EXPORT_DIR)

In [ ]:
if PROJECT_DIR.exists():
    print('Project already exists:', PROJECT_DIR)
elif GITHUB_REPO_URL:
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print('Upload a zip file that contains the fraud-detection project folder.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No project zip uploaded.')
    zip_name = next(iter(uploaded.keys()))
    upload_path = Path('/content') / zip_name
    extract_dir = Path('/content/project_upload')
    shutil.unpack_archive(str(upload_path), str(extract_dir))
    candidates = list(extract_dir.rglob('requirements-colab.txt')) + list(extract_dir.rglob('requirements.txt'))
    if not candidates:
        raise RuntimeError('Could not find a project requirements file in uploaded zip.')
    detected_project = candidates[0].parent
    shutil.copytree(detected_project, PROJECT_DIR)

assert (PROJECT_DIR / 'src' / 'train.py').exists(), 'src/train.py not found. Check project source.'
print('Project ready:', PROJECT_DIR)

## 2. Install training dependencies

This installs the lightweight Colab requirements. It excludes local services like Dagster webserver and dbt because this notebook only trains/evaluates the ML model.

In [ ]:
os.chdir(PROJECT_DIR)
req = PROJECT_DIR / 'requirements-colab.txt'
if not req.exists():
    req = PROJECT_DIR / 'requirements.txt'
print('Installing:', req)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)
print('Dependencies installed')

## 3. Configure MLflow and Kaggle access

The project uses `kagglehub.dataset_download("mlg-ulb/creditcardfraud")`. If Kaggle asks for credentials, upload `kaggle.json` from your Kaggle account.

MLflow artifacts are stored locally inside the Colab project folder and then zipped to Google Drive.

In [ ]:
# Optional Kaggle credential upload. Run this cell only if Kaggle download fails later.
UPLOAD_KAGGLE_JSON = False

if UPLOAD_KAGGLE_JSON:
    uploaded = files.upload()
    kaggle_json = next((name for name in uploaded if name == 'kaggle.json'), None)
    if not kaggle_json:
        raise RuntimeError('Upload must include kaggle.json')
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy('/content/kaggle.json', kaggle_dir / 'kaggle.json')
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
    print('Kaggle credentials installed')

os.environ['MLFLOW_TRACKING_URI'] = f"file://{PROJECT_DIR / 'mlruns'}"
# Keep Colab robust: log MLflow model artifacts, but do not force local registry creation.
os.environ['MLFLOW_REGISTER_MODEL'] = 'false'
print('MLFLOW_TRACKING_URI =', os.environ['MLFLOW_TRACKING_URI'])

## 4. Run training

This calls the same `run_training_pipeline()` used locally. It will:
- download/cache Kaggle data under `data/raw/creditcard.csv`
- clean and feature-engineer data
- run stratified train/validation/test split
- train models with scaler + SMOTE inside CV
- optimize decision threshold on validation
- evaluate once on untouched test set
- save artifacts in `models/`

In [ ]:
import json
import time

from src.train import run_training_pipeline

start = time.time()
training_result = run_training_pipeline()
elapsed = time.time() - start

print('Training finished in minutes:', round(elapsed / 60, 2))
print('Best model:', training_result['best_model'])
print(json.dumps(training_result['metrics'], indent=2)[:4000])

## 5. Run standalone evaluation

This confirms that `src.evaluate` can reload the saved model and saved threshold.

In [ ]:
from src.evaluate import run_evaluation

eval_metrics = run_evaluation()
print(json.dumps(eval_metrics, indent=2)[:4000])

## 6. Create report figures

These plots are saved into `figures/` and can be used in the final academic report.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

figures_dir = PROJECT_DIR / 'figures'
figures_dir.mkdir(exist_ok=True)

metrics_path = PROJECT_DIR / 'models' / 'metrics.json'
threshold_path = PROJECT_DIR / 'models' / 'decision_threshold.json'
metrics = json.loads(metrics_path.read_text())
threshold_payload = json.loads(threshold_path.read_text()) if threshold_path.exists() else {}

cm = metrics['confusion_matrix']
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.savefig(figures_dir / 'colab_confusion_matrix.png', dpi=180)
plt.show()

selected_metrics = ['precision', 'recall', 'f1_score', 'pr_auc', 'roc_auc', 'balanced_accuracy', 'mcc']
rows = [{'metric': m, 'value': metrics.get(m)} for m in selected_metrics if m in metrics]
df_metrics = pd.DataFrame(rows)
plt.figure(figsize=(9, 4))
sns.barplot(data=df_metrics, x='metric', y='value')
plt.xticks(rotation=35, ha='right')
plt.title('Main Imbalanced Classification Metrics')
plt.tight_layout()
plt.savefig(figures_dir / 'colab_main_metrics.png', dpi=180)
plt.show()

print('Decision threshold:', threshold_payload.get('decision_threshold'))
print('Figures saved in:', figures_dir)

## 7. Export artifacts and documentation

This creates two zip files in Google Drive:
- model artifacts zip: copy back into local `models/`
- documentation bundle zip: report inputs, docs, figures, contracts, README, notebook

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
staging = Path('/content/fraud_export_staging')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)

def copy_if_exists(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        print('Missing, skipped:', src)
        return
    if src.is_dir():
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', 'target', 'logs', '*.duckdb', '*.csv'))
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# Model artifacts for local API usage.
model_stage = staging / 'model_artifacts'
model_stage.mkdir()
for name in ['best_model.joblib', 'scaler.joblib', 'feature_columns.json', 'metrics.json', 'decision_threshold.json', 'reference_stats.json']:
    copy_if_exists(PROJECT_DIR / 'models' / name, model_stage / name)

# Documentation/report bundle.
doc_stage = staging / 'documentation_bundle'
copy_if_exists(PROJECT_DIR / 'docs', doc_stage / 'docs')
copy_if_exists(PROJECT_DIR / 'figures', doc_stage / 'figures')
copy_if_exists(PROJECT_DIR / 'contracts', doc_stage / 'contracts')
copy_if_exists(PROJECT_DIR / 'dbt_fraud' / 'models', doc_stage / 'dbt_fraud' / 'models')
for name in ['README.md', 'QUICK_START.md', 'requirements-colab.txt', 'requirements.txt']:
    copy_if_exists(PROJECT_DIR / name, doc_stage / name)
copy_if_exists(PROJECT_DIR / 'notebooks' / '06_colab_training_evaluation.ipynb', doc_stage / 'notebooks' / '06_colab_training_evaluation.ipynb')

model_zip = shutil.make_archive(str(EXPORT_DIR / f'fraud_model_artifacts_{timestamp}'), 'zip', model_stage)
doc_zip = shutil.make_archive(str(EXPORT_DIR / f'fraud_documentation_bundle_{timestamp}'), 'zip', doc_stage)

print('Created:')
print(model_zip)
print(doc_zip)

## 8. Optional direct download from Colab

If Drive sync is slow, uncomment the download lines below.

In [ ]:
# files.download(model_zip)
# files.download(doc_zip)
print('Artifacts are available in:', EXPORT_DIR)